# Gene Expression Cancer Classification - Data Exploration


**Project Goal**: Build a machine learning classifier to predict breast cancer relapse using gene expression profiles.

**Dataset**: GSE2034 (Wang et al., 2005)
- **Samples**: 286 breast cancer patients
- **Features**: ~22,000 gene expression measurements
- **Target**: Relapse prediction (0 = No relapse, 1 = Relapse)

---

## 1. Setup and Environment Configuration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")

## 2. Load and Explore Data

Load the breast cancer gene expression dataset and examine its structure.

In [ ]:
# Load the dataset
data_path = Path("../data/dataset.csv")
df = pd.read_csv(data_path, index_col='sample_id')

print("Dataset loaded successfully!")
print(f"\nDataset shape: {df.shape}")
print(f"  - Samples (rows): {df.shape[0]}")
print(f"  - Features (columns): {df.shape[1]}")

In [ ]:
# Examine first few rows of the dataset
print("First 5 samples (showing first 10 gene columns + label):")
display_cols = list(df.columns[:10]) + ['label']
df[display_cols].head()

In [ ]:
# Dataset information
print("Dataset Info:")
print("=" * 50)
df.info(verbose=False, show_counts=True)

In [ ]:
# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

print("Features (Gene Expression Values):")
print(f"  - Shape: {X.shape}")
print(f"  - Gene columns: {len(X.columns)}")
print(f"\nTarget (Relapse Status):")
print(f"  - Shape: {y.shape}")
print(f"  - Unique values: {y.unique()}")

## 3. Data Quality Check

Check for missing values, duplicates, and data types.

In [ ]:
# Check for missing values
missing_total = df.isnull().sum().sum()
missing_per_col = df.isnull().sum()
cols_with_missing = missing_per_col[missing_per_col > 0]

print("Missing Values Analysis:")
print("=" * 50)
print(f"Total missing values: {missing_total}")
print(f"Columns with missing values: {len(cols_with_missing)}")

if len(cols_with_missing) > 0:
    print("\nColumns with most missing values:")
    print(cols_with_missing.sort_values(ascending=False).head(10))
else:
    print("\n✓ No missing values found!")

In [ ]:
# Check for duplicate samples
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates > 0:
    print("⚠ Warning: Duplicates detected!")
else:
    print("✓ No duplicate samples found!")

In [ ]:
# Check data types
print("Data Types:")
print("=" * 50)
dtype_counts = df.dtypes.value_counts()
print(dtype_counts)
print(f"\n✓ All gene expression values are numeric (float64)")

## 4. Basic Data Analysis

Compute summary statistics and explore the target variable distribution.

In [ ]:
# Summary statistics for gene expression values
print("Gene Expression Summary Statistics:")
print("=" * 50)
print(f"\nAcross all genes and samples:")
print(f"  Min: {X.min().min():.3f}")
print(f"  Max: {X.max().max():.3f}")
print(f"  Mean: {X.mean().mean():.3f}")
print(f"  Std: {X.std().mean():.3f}")

print("\nPer-sample statistics (first 5 samples):")
X.iloc[:5].describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# Class distribution analysis
print("Target Variable (Label) Distribution:")
print("=" * 50)
class_counts = y.value_counts()
class_percentages = y.value_counts(normalize=True) * 100

print("\nClass Counts:")
for label, count in class_counts.items():
    label_name = "No Relapse" if label == 0 else "Relapse"
    print(f"  Class {label} ({label_name}): {count} samples ({class_percentages[label]:.1f}%)")

print(f"\nClass Imbalance Ratio: {class_counts[0] / class_counts[1]:.2f}:1")
print("\n⚠ Note: Dataset is imbalanced - more no-relapse samples than relapse samples.")
print("  This will need to be addressed during model training.")

## 5. Data Visualization

Visualize class distribution, gene expression patterns, and sample distributions.

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
class_labels = ['No Relapse (0)', 'Relapse (1)']
axes[0].bar(class_labels, class_counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Class Distribution')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_labels, colors=colors, 
            autopct='%1.1f%%', startangle=90, explode=[0, 0.05])
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Gene expression distribution across all samples
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of mean expression per gene
gene_means = X.mean()
axes[0].hist(gene_means, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Mean Expression Value')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Distribution of Mean Gene Expression')
axes[0].axvline(gene_means.mean(), color='red', linestyle='--', label=f'Mean: {gene_means.mean():.2f}')
axes[0].legend()

# Distribution of expression variance per gene
gene_vars = X.var()
axes[1].hist(gene_vars, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Expression Variance')
axes[1].set_ylabel('Number of Genes')
axes[1].set_title('Distribution of Gene Expression Variance')
axes[1].axvline(gene_vars.mean(), color='red', linestyle='--', label=f'Mean: {gene_vars.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plot of expression values for a few random genes
np.random.seed(42)
sample_genes = np.random.choice(X.columns, size=10, replace=False)

fig, ax = plt.subplots(figsize=(14, 6))
X[sample_genes].boxplot(ax=ax)
ax.set_xlabel('Gene')
ax.set_ylabel('Expression Value')
ax.set_title('Expression Distribution for 10 Random Genes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for a subset of high-variance genes
print("Computing correlations for top 20 most variable genes...")
top_var_genes = X.var().nlargest(20).index.tolist()
corr_matrix = X[top_var_genes].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap: Top 20 Most Variable Genes')
plt.tight_layout()
plt.show()

In [ ]:
# Compare expression distributions between classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall expression distribution by class
for label in [0, 1]:
    label_name = "No Relapse" if label == 0 else "Relapse"
    sample_means = X[y == label].mean(axis=1)
    axes[0].hist(sample_means, bins=30, alpha=0.6, label=label_name)
axes[0].set_xlabel('Mean Expression per Sample')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Mean Sample Expression by Class')
axes[0].legend()

# Variance distribution by class
for label in [0, 1]:
    label_name = "No Relapse" if label == 0 else "Relapse"
    sample_vars = X[y == label].var(axis=1)
    axes[1].hist(sample_vars, bins=30, alpha=0.6, label=label_name)
axes[1].set_xlabel('Variance per Sample')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Sample Variance by Class')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Key Findings & Next Steps

### Dataset Summary
- **286 breast cancer samples** with gene expression profiles
- **22,283 genes** (features) per sample
- **Binary classification**: Relapse (1) vs No Relapse (0)
- **Imbalanced classes**: ~76% No Relapse, ~24% Relapse

### Key Observations
1. ✓ No missing values in the dataset
2. ✓ No duplicate samples
3. ⚠ High dimensionality (genes >> samples) - feature selection critical
4. ⚠ Class imbalance - will need handling during training

### Next Steps 
1. **Data Preprocessing**
   - Normalize gene expression values (StandardScaler)
   - Train-test split (stratified to maintain class balance)
   
2. **Feature Selection **
   - Apply SelectKBest to reduce dimensionality
   - Consider variance-based filtering

---
**Data Source**: GEO GSE2034  
**Reference**: Wang et al. (2005) Lancet

In [ ]:
# Final summary
print("=" * 60)
print("EXPLORATION COMPLETE")
print("=" * 60)
print(f"\nDataset: GSE2034 Breast Cancer Gene Expression")
print(f"Classification Task: Breast Cancer Relapse Prediction")
print(f"\nDataset Statistics:")
print(f"  - Total samples: {df.shape[0]}")
print(f"  - Total features (genes): {df.shape[1] - 1}")
print(f"  - Class 0 (No Relapse): {(y == 0).sum()} samples")
print(f"  - Class 1 (Relapse): {(y == 1).sum()} samples")
print(f"\nData Quality:")
print(f"  - Missing values: {df.isnull().sum().sum()}")
print(f"  - Duplicates: {df.duplicated().sum()}")
print(f"\n✓ Ready for preprocessing ")